# Final cohort summary — `ours_di`

Standalone notebook that regenerates the locked cohort numbers directly from the materialized
`ours_di` dataset, using the **same `dsm` pipeline the experiments use**, so the numbers match
exactly (same label rule, same SMILES/ICD filter, same temporal split).

Run from the repo root in the project env, e.g. `uv run jupyter lab`.

In [1]:
import pandas as pd
from dsm.experiments import DATASETS
from dsm.datasets import materialize

spec = DATASETS["ours_di"]
cohort = pd.read_parquet(materialize(spec))   # the exact cohort the experiments train/test on
len(cohort)

14134

## Final locked numbers

In [2]:
total = len(cohort)
n_pos = int(cohort["label"].sum())          # approved
n_neg = total - n_pos                          # failed

test = cohort[cohort["split"] == "test"]
n_test, n_test_pos = len(test), int(test["label"].sum())

n_drugs = cohort["drugbank_id"].nunique()
n_indications = cohort["indication"].nunique()
avg_ind_per_drug = cohort.groupby("drugbank_id")["indication"].nunique().mean()

print("=== Final cohort summary (ours_di) ===")
print(f"total drug-indication pairs : {total:,}")
print(f"  approved (label=1)        : {n_pos:,}  ({n_pos / total:.1%})")
print(f"  failed   (label=0)        : {n_neg:,}  ({n_neg / total:.1%})")
print(f"test set                    : {n_test:,} pairs, {n_test_pos:,} positives")
print(f"unique drugs                : {n_drugs:,}")
print(f"unique indications          : {n_indications:,}")
print(f"avg indications per drug    : {avg_ind_per_drug:.2f}")

=== Final cohort summary (ours_di) ===
total drug-indication pairs : 14,134
  approved (label=1)        : 2,469  (17.5%)
  failed   (label=0)        : 11,665  (82.5%)
test set                    : 2,637 pairs, 624 positives
unique drugs                : 2,313
unique indications          : 4,190
avg indications per drug    : 5.18


## SMILES coverage / filtering impact

The funnel from the raw candidate table to the final cohort, reproduced with the same `dsm`
helpers (`apply_label`, `_row_filter`) the pipeline uses. The big cuts are label-undecidable rows
and missing ICD codes — SMILES coverage is essentially complete.

In [3]:
from dsm.dataset import load_candidate_detail, apply_label
from dsm.datasets import _smiles_list, _flat_icd, _row_filter
from dsm.config import ModelingConfig

src = load_candidate_detail(ModelingConfig().candidate_detail_path)
n_raw = len(src)

src = apply_label(src, spec.label).reset_index(drop=True)   # keep rows with a decidable outcome
n_labeled = len(src)

smiles_col = "smiles_canonical" if "smiles_canonical" in src.columns else "smiles"
smiles = src[smiles_col].map(_smiles_list)
icd = src.get("icd10_codes", pd.Series([None] * len(src))).map(_flat_icd)
has_smiles = smiles.map(len) > 0

keep, n_no_smiles, n_no_icd = _row_filter(smiles, icd)   # drop empty SMILES or empty ICD
n_kept = int(keep.sum())
n_no_year = n_kept - total   # passed SMILES+ICD but no usable date for the temporal split

print("=== SMILES coverage / filtering impact ===")
print(f"raw candidate rows             : {n_raw:,}")
print(f"with a decidable label         : {n_labeled:,}  (dropped {n_raw - n_labeled:,} undecidable)")
print(f"SMILES coverage (labeled rows) : {has_smiles.mean():.2%}  ({int(has_smiles.sum()):,} with SMILES)")
print(f"dropped — no SMILES            : {n_no_smiles:,}")
print(f"dropped — no ICD (had SMILES)  : {n_no_icd:,}")
print(f"after SMILES+ICD filter        : {n_kept:,}")
print(f"dropped — no temporal year     : {n_no_year:,}")
print(f"final cohort                   : {total:,}")

=== SMILES coverage / filtering impact ===
raw candidate rows             : 55,350
with a decidable label         : 31,977  (dropped 23,373 undecidable)
SMILES coverage (labeled rows) : 99.98%  (31,971 with SMILES)
dropped — no SMILES            : 6
dropped — no ICD (had SMILES)  : 17,837
after SMILES+ICD filter        : 14,134
dropped — no temporal year     : 0
final cohort                   : 14,134
